this code relates to extraction and analysis of audio feature, done in the scope of an experiment on musical improvisation, mental states and time perception.

---------- EXPERIMENT DESCRIPTION ------------

1. Research Question - does musical improvisation affect musicians time perception? 2 dimensions of improvisation: Group vs solo; familiar harmony (A) vs unfamiliar harmony (B) vs Free (C). 6 tasks per participant. groups included 4 musicians
2. Data Collection - answers about time perception enjoyment, difficulty etc + recordings of improvisations
3. Feature Extraction - at a point in the analysis data suggested musical features could be driving changes in perceived duration - musical tempo, and how clear it could be perceived could be possible factors. To confirm this we extracted BPM and beat confidence using Essentia
4. Statistical Analysis - first simple T-tests and one-way anovas compared bpm and beat confidence across the involved musical conditions. Later, in full paper, we used Linear-Mixed models to see if these audio features predicted Time perception.
5. Results - there were indeed differences in beat confidence, not BPM, but in fact these did not predict duration estimates



In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
from sklearn.linear_model import LinearRegression

from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from scipy.stats import shapiro
from scipy.stats import mannwhitneyu

#for mixed models and anovas
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pingouin as pg

#for MIR
import os
import essentia.standard as es

In [4]:
# loading dataset with participants's answers for each experimental trial

df = pd.read_csv('/Users/ligia/Library/Mobile Documents/com~apple~CloudDocs/outros projetos/improvisação/results/improv-time 2 - clean_columns_added_relative_estimates.csv', sep = ",")
df.head()

,Unnamed: 0,data_collection,ID de participante,Tipo de instrumento,G01Q06. Escolha o ID da sessão,Assigned_group,Group-Solo,Type-improv,Real_duration(s),Duration_estimate[min.],...,FA,ED,MW,MB,TRI,Probe2. Indique. por favor. qual dos seguintes estados mentais foi mais predominante neste take:,Formato audio,age,sex,relative_estimate
0,0,PJ,LA06H,Harmonic,Grupo - B - Não familiar,Group0,Group,B,130,3,...,4.5,1.0,1.5,2.0,3.0,Estive focado na performance musical.,.mp3,26.0,Masculino,1.538462
1,1,PJ,oa03B,Bass,Grupo - B - Não familiar,Group0,Group,B,130,3,...,4.0,1.0,1.7,3.0,1.5,Estive focado na performance musical.,.mp3,35.0,Masculino,1.538462
2,2,PJ,oa07D,Drum,Grupo - B - Não familiar,Group0,Group,B,130,3,...,4.5,1.0,1.0,5.0,1.0,Estive focado na performance musical.,.mp3,29.0,Masculino,1.669231
3,3,PJ,OA05M,Melodic,Grupo - B - Não familiar,Group0,Group,B,130,2,...,3.6,1.6,1.7,1.8,2.2,Estive focado na performance musical.,.mp3,26.0,Masculino,1.369231
4,4,PJ,ea09D,Drum,Grupo - B - Não familiar,Group1,Group,B,170,3,...,4.9,1.2,1.3,1.0,1.1,Estive focado na performance musical.,.mp3,29.0,Masculino,1.058824


In [5]:
# confirming variables, represented as columns in the dataset, where accurately defined as categorical or continuous
df.dtypes

Unnamed: 0                                                                                            int64
data_collection                                                                                      object
ID de participante                                                                                   object
Tipo de instrumento                                                                                  object
G01Q06. Escolha o ID da sessão                                                                       object
Assigned_group                                                                                       object
Group-Solo                                                                                           object
Type-improv                                                                                          object
Real_duration(s)                                                                                      int64
Duration_estimate[min.]     

In [ ]:
# add path for each audio file into csv. Audio files were labelled according to the experimental condition, which can be extracted by the variables in each row/trial

df["recording_path"] = ""

for index, row in df.iterrows():
    if (row["Group-Solo"] == "Group"):
        df.loc[index, "recording_path"] = "/Users/ligia/Library/Mobile Documents/com~apple~CloudDocs/outros projetos/improvisação/gravações/Groups/" + row["Assigned_group"] + " " + row["Type-improv"] + " " + row["Formato audio"]
        

for index, row in df.iterrows():
    if (row["Group-Solo"] == "Solo"):
        df.loc[index, "recording_path"] = "/Users/ligia/Library/Mobile Documents/com~apple~CloudDocs/outros projetos/improvisação/gravações/Solos/" + row["ID de participante"] + " Solo " + row["Type-improv"] + " " + row["Formato audio"]

df.to_csv("improv-time 3 - added recording paths.csv")

In [ ]:
#    estimate BPM and beat confidence, and store values in the csv, in new columns, and also as arrays for statistical comparison in this same file 


import os
import subprocess
import tempfile
import numpy as np
import essentia.standard as es

# 2 separate folders of recordings: group and solos
paths = (
    '/Users/ligia/Library/Mobile Documents/com~apple~CloudDocs/outros projetos/improvisação/gravações/Groups',
    '/Users/ligia/Library/Mobile Documents/com~apple~CloudDocs/outros projetos/improvisação/gravações/Solos'
)


beat_confidence_groups = []
beat_confidence_solos = []
bpm_groups = []
bpm_solos = []

df["BPM_estimated"] = ""
df["beat_confidence"] = ""

# ---------------------------------------------------------
# FUNCTION: convert any audio file to standardized WAV
# mono / 44.1kHz / PCM 16-bit
# ---------------------------------------------------------

def standardize_audio(input_path):

    temp_wav = tempfile.NamedTemporaryFile(
        suffix=".wav",
        delete=False
    )

    output_path = temp_wav.name
    temp_wav.close()

    command = [
        "ffmpeg",
        "-y",
        "-i", input_path,
        "-ac", "1",
        "-ar", "44100",
        "-sample_fmt", "s16",
        output_path
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    print("\n-----------------------------------")
    print("FFMPEG INPUT :", input_path)
    print("FFMPEG OUTPUT:", output_path)
    print("RETURN CODE  :", result.returncode)

    if result.returncode != 0:
        print("\nFFMPEG STDERR:")
        print(result.stderr)

    if os.path.exists(output_path):
        print("OUTPUT EXISTS:", True)
        print("OUTPUT SIZE  :", os.path.getsize(output_path), "bytes")
    else:
        print("OUTPUT EXISTS:", False)

    print("-----------------------------------\n")

    return output_path


# ---------------------------------------------------------
# ANALYZE GROUPS
# ---------------------------------------------------------

directory = os.fsencode(paths[0])

for index, row in df.iterrows():

    full_path = df.loc[index, "recording_path"]

    print("\n===================================")
    print("ROW INDEX:", index)
    print("SOURCE FILE:", full_path)
    print("FILE EXISTS:", os.path.exists(full_path))

    if os.path.exists(full_path):
        print("FILE SIZE:", os.path.getsize(full_path), "bytes")

    print("===================================\n")

    try:

        # ---------------------------------------------
        # STANDARDIZE AUDIO
        # ---------------------------------------------
        standardized_file = standardize_audio(full_path)

        print("TEMP FILE:", standardized_file)
        print("TEMP EXISTS:", os.path.exists(standardized_file))

        # ---------------------------------------------
        # LOAD STANDARDIZED AUDIO
        # ---------------------------------------------
        audio = es.MonoLoader(
            filename=standardized_file,
            sampleRate=44100
        )()

        print("shape:", audio.shape)
        print("min/max:", np.min(audio), np.max(audio))
        print("nan:", np.isnan(audio).any())

        # ---------------------------------------------
        # RHYTHM EXTRACTION
        # ---------------------------------------------
        rhythm_extractor = es.RhythmExtractor2013(
            method="multifeature"
        )

        bpm, beats, beat_confidence, _, _ = rhythm_extractor(audio)

        print(
            "bpm =", bpm,
            "beat confidence =", beat_confidence
        )

        beat_confidence_groups.append(float(beat_confidence))
        bpm_groups.append(float(bpm))

        df.loc[index, "BPM_estimated"] = bpm
        df.loc[index, "beat_confidence"] = beat_confidence

        # ---------------------------------------------
        # DELETE TEMP FILE
        # ---------------------------------------------
        os.remove(standardized_file)

    except Exception as e:

        print("ERROR:", e)

        if 'standardized_file' in locals():
            print(
                "TEMP FILE EXISTS:",
                os.path.exists(standardized_file)
            )

            if os.path.exists(standardized_file):
                print(
                    "TEMP FILE SIZE:",
                    os.path.getsize(standardized_file),
                    "bytes"
                )

# ---------------------------------------------------------
# ANALYZE SOLOS
# ---------------------------------------------------------

directory = os.fsencode(paths[1])

for index, row in df.iterrows():

    full_path = df.loc[index, "recording_path"]

    print("\n===================================")
    print("ROW INDEX:", index)
    print("SOURCE FILE:", full_path)
    print("FILE EXISTS:", os.path.exists(full_path))

    if os.path.exists(full_path):
        print("FILE SIZE:", os.path.getsize(full_path), "bytes")

    print("===================================\n")

    try:

        # ---------------------------------------------
        # STANDARDIZE AUDIO
        # ---------------------------------------------
        standardized_file = standardize_audio(full_path)

        print("TEMP FILE:", standardized_file)
        print("TEMP EXISTS:", os.path.exists(standardized_file))

        # ---------------------------------------------
        # LOAD STANDARDIZED AUDIO
        # ---------------------------------------------
        audio = es.MonoLoader(
            filename=standardized_file,
            sampleRate=44100
        )()

        print("shape:", audio.shape)
        print("min/max:", np.min(audio), np.max(audio))
        print("nan:", np.isnan(audio).any())

        # ---------------------------------------------
        # RHYTHM EXTRACTION
        # ---------------------------------------------
        rhythm_extractor = es.RhythmExtractor2013(
            method="multifeature"
        )

        bpm, beats, beat_confidence, _, _ = rhythm_extractor(audio)

        beat_confidence = max(0.0, beat_confidence)

        print(
            "bpm =", bpm,
            "beat confidence =", beat_confidence
        )

        beat_confidence_solos.append(float(beat_confidence))
        bpm_solos.append(float(bpm))

        df.loc[index, "BPM_estimated"] = bpm
        df.loc[index, "beat_confidence"] = beat_confidence

        # ---------------------------------------------
        # DELETE TEMP FILE
        # ---------------------------------------------
        os.remove(standardized_file)

    except Exception as e:

        print("ERROR:", e)

        if 'standardized_file' in locals():
            print(
                "TEMP FILE EXISTS:",
                os.path.exists(standardized_file)
            )

            if os.path.exists(standardized_file):
                print(
                    "TEMP FILE SIZE:",
                    os.path.getsize(standardized_file),
                    "bytes"
                )



ROW INDEX: 0
SOURCE FILE: /Users/ligia/Library/Mobile Documents/com~apple~CloudDocs/outros projetos/improvisação/gravações/Groups/Group0 B .mp3
FILE EXISTS: True
FILE SIZE: 3286121 bytes


-----------------------------------
FFMPEG INPUT : /Users/ligia/Library/Mobile Documents/com~apple~CloudDocs/outros projetos/improvisação/gravações/Groups/Group0 B .mp3
FFMPEG OUTPUT: /var/folders/df/24q1vg6s2dd3v6ttwc3nn0qw0000gn/T/tmpre7ok_eq.wav
RETURN CODE  : 0
OUTPUT EXISTS: True
OUTPUT SIZE  : 12061518 bytes
-----------------------------------

TEMP FILE: /var/folders/df/24q1vg6s2dd3v6ttwc3nn0qw0000gn/T/tmpre7ok_eq.wav
TEMP EXISTS: True
shape: (6030720,)
min/max: -0.8171387 0.98913574
nan: False
bpm = 120.73987579345703 beat confidence = 0.9138050079345703

ROW INDEX: 1
SOURCE FILE: /Users/ligia/Library/Mobile Documents/com~apple~CloudDocs/outros projetos/improvisação/gravações/Groups/Group0 B .mp3
FILE EXISTS: True
FILE SIZE: 3286121 bytes


-----------------------------------
FFM

In [8]:
df.to_csv("improv-time results 4 - final.csv")

In [ ]:
# see if bpm and beat confidence are signif different in group vs solo playing by running a T test - no normal distribution or variance


u, p = mannwhitneyu(
    beat_confidence_groups,
    beat_confidence_solos,
    alternative="two-sided"
)

print(f"beat confidence U = {u:.3f}, p = {p:.4f}")

groups_mean = np.mean(beat_confidence_groups)
groups_sd   = np.std(beat_confidence_groups, ddof=1)

solos_mean = np.mean(beat_confidence_solos)
solos_sd   = np.std(beat_confidence_solos, ddof=1)

print(f"beat confidence Groups: mean = {groups_mean:.3f}, SD = {groups_sd:.3f}, n = {len(beat_confidence_groups)}")
print(f"beat confidence Solos : mean = {solos_mean:.3f}, SD = {solos_sd:.3f}, n = {len(beat_confidence_solos)}")

u, p = mannwhitneyu(
    bpm_groups,
    bpm_solos,
    alternative="two-sided"
)

print(f"bpm U = {u:.3f}, p = {p:.4f}")

groups_mean = np.mean(bpm_groups)
groups_sd   = np.std(bpm_groups, ddof=1)

solos_mean = np.mean(bpm_solos)
solos_sd   = np.std(bpm_solos, ddof=1)

print(f"bpm Groups: mean = {groups_mean:.3f}, SD = {groups_sd:.3f}, n = {len(bpm_groups)}")
print(f"bpm Solos : mean = {solos_mean:.3f}, SD = {solos_sd:.3f}, n = {len(bpm_solos)}")

"""this analysis actually had a problem of including 4x time group recordings than we actually had, as it included in the array extracted values 4x when running through the csv and executing the extraction for each of the 4 musicians in the same group. A better approach is the use of LMM using the newly created bpm and beat confidence columns, done in another file. this block stays here as a proof of concept, and for future use if needed
Without this 4x group inclusion there was actually a significant difference between group and solos conditions"""

beat confidence U = 36153.000, p = 0.3015
beat confidence Groups: mean = 0.589, SD = 0.711, n = 276
beat confidence Solos : mean = 0.605, SD = 0.697, n = 276
bpm U = 38072.000, p = 0.9934
bpm Groups: mean = 125.286, SD = 20.669, n = 276
bpm Solos : mean = 125.287, SD = 20.669, n = 276


'yes they were but beat confidence may be more important than bpm as too small bpm difference Hammerschmidt et al., 2021 -Disco Time: The Relationship Between Perceived Duration and Tempo in Music \n\nHammerschmidt et al, 2021:\n\nDuration reproductions increased with faster tempo, no effect on verbal. but changes in tempo only affected reproduction estimates if higher than 20 bpm (comparing 125 to 105bpm, intermediate comparisons between these and 115 showed no effects)'

In [10]:
# -------------- Group required more FA than solo, but this did not impact DE, beat confidence appears to have impacted DE. Free condition also led to more FA than the structured ones, but this did not impact DE. So, we would expect beat confidence to not change between free and structured, otherwise we would see an effect in DE there...  2 separate folders of recordings: free and structured
paths = (
    '/Users/ligia/Library/Mobile Documents/com~apple~CloudDocs/outros projetos/improvisação/gravações/all recordings, split by improv task/Free',
    '/Users/ligia/Library/Mobile Documents/com~apple~CloudDocs/outros projetos/improvisação/gravações/all recordings, split by improv task/familiar',
    '/Users/ligia/Library/Mobile Documents/com~apple~CloudDocs/outros projetos/improvisação/gravações/all recordings, split by improv task/Unfamiliar'
)

beat_confidence_free = []
beat_confidence_familiar = []
beat_confidence_unfamiliar = []
bpm_free = []
bpm_familiar = []
bpm_unfamiliar = []

# ---------------------------------------------------------
# FUNCTION: convert any audio file to standardized WAV
# mono / 44.1kHz / PCM 16-bit
# ---------------------------------------------------------

def standardize_audio(input_path):

    temp_wav = tempfile.NamedTemporaryFile(
        suffix=".wav",
        delete=False
    )

    output_path = temp_wav.name
    temp_wav.close()

    command = [
        "ffmpeg",
        "-y",                 # overwrite temp file
        "-i", input_path,     # input file
        "-ac", "1",           # mono
        "-ar", "44100",       # sample rate
        "-sample_fmt", "s16", # PCM 16-bit
        output_path
    ]

    subprocess.run(
        command,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    return output_path


# ---------------------------------------------------------
# ANALYZE FREE
# ---------------------------------------------------------

directory = os.fsencode(paths[0])

for file in os.listdir(directory):

    filename = os.fsdecode(file)

    if filename.lower().endswith(
        (".wav", ".aif", ".aiff", ".mp3", ".m4a")
    ):

        full_path = os.path.join(paths[0], filename)

        print("\n" + filename)

        try:

            # ---------------------------------------------
            # STANDARDIZE AUDIO
            # ---------------------------------------------
            standardized_file = standardize_audio(full_path)

            # ---------------------------------------------
            # LOAD STANDARDIZED AUDIO
            # ---------------------------------------------
            audio = es.MonoLoader(
                filename=standardized_file,
                sampleRate=44100
            )()

            print("shape:", audio.shape)
            print("min/max:", np.min(audio), np.max(audio))
            print("nan:", np.isnan(audio).any())

            # ---------------------------------------------
            # RHYTHM EXTRACTION
            # ---------------------------------------------
            rhythm_extractor = es.RhythmExtractor2013(
                method="multifeature"
            )

            bpm, beats, beat_confidence, _, _ = rhythm_extractor(audio)

            # clamp tiny negative float artifacts
            beat_confidence = max(0.0, beat_confidence)

            print(
                "bpm = ", bpm,
                "beat confidence = ", beat_confidence
            )

            beat_confidence_free.append(float(beat_confidence))
            bpm_free.append(float(bpm))

            # ---------------------------------------------
            # DELETE TEMP FILE
            # ---------------------------------------------
            os.remove(standardized_file)

        except Exception as e:

            print("ERROR:", e)
# ---------------------------------------------------------
# ANALYZE FAMILIAR
# ---------------------------------------------------------

directory = os.fsencode(paths[1])

for file in os.listdir(directory):

    filename = os.fsdecode(file)

    if filename.lower().endswith(
        (".wav", ".aif", ".aiff", ".mp3", ".m4a")
    ):

        full_path = os.path.join(paths[1], filename)

        print("\n" + filename)

        try:

            # ---------------------------------------------
            # STANDARDIZE AUDIO
            # ---------------------------------------------
            standardized_file = standardize_audio(full_path)

            # ---------------------------------------------
            # LOAD STANDARDIZED AUDIO
            # ---------------------------------------------
            audio = es.MonoLoader(
                filename=standardized_file,
                sampleRate=44100
            )()

            print("shape:", audio.shape)
            print("min/max:", np.min(audio), np.max(audio))
            print("nan:", np.isnan(audio).any())

            # ---------------------------------------------
            # RHYTHM EXTRACTION
            # ---------------------------------------------
            rhythm_extractor = es.RhythmExtractor2013(
                method="multifeature"
            )

            bpm, beats, beat_confidence, _, _ = rhythm_extractor(audio)

            # clamp tiny negative float artifacts
            beat_confidence = max(0.0, beat_confidence)

            print(
                "bpm = ", bpm,
                "beat confidence = ", beat_confidence
            )

            beat_confidence_familiar.append(float(beat_confidence))
            bpm_familiar.append(float(bpm))

            # ---------------------------------------------
            # DELETE TEMP FILE
            # ---------------------------------------------
            os.remove(standardized_file)

        except Exception as e:

            print("ERROR:", e)

# ---------------------------------------------------------
# ANALYZE UNFAMILIAR
# ---------------------------------------------------------

directory = os.fsencode(paths[2])

for file in os.listdir(directory):

    filename = os.fsdecode(file)

    if filename.lower().endswith(
        (".wav", ".aif", ".aiff", ".mp3", ".m4a")
    ):

        full_path = os.path.join(paths[2], filename)

        print("\n" + filename)

        try:

            # ---------------------------------------------
            # STANDARDIZE AUDIO
            # ---------------------------------------------
            standardized_file = standardize_audio(full_path)

            # ---------------------------------------------
            # LOAD STANDARDIZED AUDIO
            # ---------------------------------------------
            audio = es.MonoLoader(
                filename=standardized_file,
                sampleRate=44100
            )()

            print("shape:", audio.shape)
            print("min/max:", np.min(audio), np.max(audio))
            print("nan:", np.isnan(audio).any())

            # ---------------------------------------------
            # RHYTHM EXTRACTION
            # ---------------------------------------------
            rhythm_extractor = es.RhythmExtractor2013(
                method="multifeature"
            )

            bpm, beats, beat_confidence, _, _ = rhythm_extractor(audio)

            # clamp tiny negative float artifacts
            beat_confidence = max(0.0, beat_confidence)

            print(
                "bpm = ", bpm,
                "beat confidence = ", beat_confidence
            )

            beat_confidence_unfamiliar.append(float(beat_confidence))
            bpm_unfamiliar.append(float(bpm))

            # ---------------------------------------------
            # DELETE TEMP FILE
            # ---------------------------------------------
            os.remove(standardized_file)

        except Exception as e:

            print("ERROR:", e)            


Jdrum 5 Solo C Free.m4a
shape: (5784921,)
min/max: -1.0 0.9999695
nan: False
bpm =  136.04983520507812 beat confidence =  1.4985120296478271

Jsax5 - C Free.wav
shape: (6866727,)
min/max: -0.9885254 0.9884949
nan: False
bpm =  90.41067504882812 beat confidence =  0.0

PJ2 Group 2 C.m4a
shape: (7960050,)
min/max: -1.0 0.9999695
nan: False
bpm =  122.42682647705078 beat confidence =  0.0

Jguitar8_livre130#01.wav
shape: (7408800,)
min/max: -0.03274536 0.031402588
nan: False
bpm =  135.21424865722656 beat confidence =  0.0

Jdrum7 Solo C Free.m4a
shape: (6646694,)
min/max: -1.0 0.9999695
nan: False
bpm =  107.47891998291016 beat confidence =  0.15551042556762695

Pj2 Bass1 Solo C.m4a
shape: (6221451,)
min/max: -0.72976685 0.69522095
nan: False
bpm =  126.3013916015625 beat confidence =  0.04909767955541611

Harmonic (G. 1) hh04 C Free .mp3
shape: (8064000,)
min/max: -0.9807129 0.9884033
nan: False
bpm =  178.2057647705078 beat confidence =  0.0

grupo 9 livre.m4a
shape: (7519755,)
min/ma

In [11]:
# see if bpm and beat confidence are signif different in improv conditions

from scipy.stats import f_oneway
import numpy as np

# ---------------------------------------------------------
# ONE-WAY ANOVA: BEAT CONFIDENCE
# ---------------------------------------------------------

F, p = f_oneway(
    beat_confidence_free,
    beat_confidence_familiar,
    beat_confidence_unfamiliar
)

print(f"Beat confidence ANOVA: F = {F:.3f}, p = {p:.4f}")

# Descriptives
print(f"free:        mean = {np.mean(beat_confidence_free):.3f}, SD = {np.std(beat_confidence_free, ddof=1):.3f}, n = {len(beat_confidence_free)}")
print(f"familiar:    mean = {np.mean(beat_confidence_familiar):.3f}, SD = {np.std(beat_confidence_familiar, ddof=1):.3f}, n = {len(beat_confidence_familiar)}")
print(f"unfamiliar:  mean = {np.mean(beat_confidence_unfamiliar):.3f}, SD = {np.std(beat_confidence_unfamiliar, ddof=1):.3f}, n = {len(beat_confidence_unfamiliar)}")




# ---------------------------------------------------------
# ONE-WAY ANOVA: BPM
# ---------------------------------------------------------

F, p = f_oneway(
    bpm_free,
    bpm_familiar,
    bpm_unfamiliar
)

print(f"BPM ANOVA: F = {F:.3f}, p = {p:.4f}")

# Descriptives
print(f"free:        mean = {np.mean(bpm_free):.3f}, SD = {np.std(bpm_free, ddof=1):.3f}, n = {len(bpm_free)}")
print(f"familiar:    mean = {np.mean(bpm_familiar):.3f}, SD = {np.std(bpm_familiar, ddof=1):.3f}, n = {len(bpm_familiar)}")
print(f"unfamiliar:  mean = {np.mean(bpm_unfamiliar):.3f}, SD = {np.std(bpm_unfamiliar, ddof=1):.3f}, n = {len(bpm_unfamiliar)}")

"""also, free differed in beat confidence, but this did not affect DE!!"""

Beat confidence ANOVA: F = 7.989, p = 0.0005
free:        mean = 0.197, SD = 0.383, n = 59
familiar:    mean = 0.463, SD = 0.605, n = 60
unfamiliar:  mean = 0.624, SD = 0.714, n = 58
BPM ANOVA: F = 0.426, p = 0.6538
free:        mean = 125.487, SD = 23.662, n = 59
familiar:    mean = 129.187, SD = 22.032, n = 60
unfamiliar:  mean = 126.999, SD = 20.116, n = 58


'also, free differed in beat confidence, but this did not affect DE!!'